In [0]:
# Widgets
dbutils.widgets.removeAll()
dbutils.widgets.dropdown("env", "dev", ["dev", "prod"], "Ambiente")

In [0]:
catalog = f"{dbutils.widgets.get('env')}_fraud"
gold = f"{catalog}.gold"

print(f"Construyendo Modelo Estrella en {gold}...")

# DIMENSIÓN: Clientes (Atributos descriptivos)
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {gold}.dim_clientes (
    id_cliente BIGINT,
    nombre STRING,
    rango_edad STRING,
    rango_fico STRING,
    estado STRING,
    ingreso_anual DOUBLE
) USING DELTA
""")

# DIMENSIÓN: Tarjetas
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {gold}.dim_tarjetas (
    id_tarjeta BIGINT,
    tipo_tarjeta STRING,
    marca_tarjeta STRING,
    limite_credito DOUBLE
) USING DELTA
""")

# DIMENSIÓN: Catálogo MCC (Comercios)
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {gold}.dim_mcc (
    codigo_mcc INT,
    descripcion STRING,
    categoria_grupo STRING
) USING DELTA
""")

# TABLA DE HECHOS: Transacciones (Solo llaves y métricas)
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {gold}.fact_transacciones (
    id_transaccion_sk STRING, -- Llave subrogada o compuesta
    fecha_completa TIMESTAMP,
    id_cliente BIGINT,
    id_tarjeta BIGINT,
    codigo_mcc INT,
    monto DOUBLE,
    es_fraude INT,
    hora_pico BOOLEAN,
    dia_semana STRING,
    process_date STRING
) USING DELTA
""")